# ファウンデーション モデルのファインチューニングとデプロイ

**学習目標**

1. Agent Platform 上で LLM をファインチューニングする方法を学ぶ
1. チューニング済み LLM をデプロイしてクエリを実行する方法を学ぶ
1. チューニング済み LLM を評価する方法を学ぶ

LLM のゼロからの作成には、膨大なデータ、多大なコンピューティング リソース、高度な専門知識が必要です。このノートブックでは、Agent Platform Studio で Gemini ファウンデーション モデルをファインチューニングし、より特定のタスクや知識ドメインに合わせてカスタマイズする方法を学びます。
プロンプト設計は迅速な実験には優れていますが、トレーニング データが利用可能な場合は、モデルをチューニングすることでより高い品質を実現できます。モデルのチューニングにより、実行させたいタスクの例に基づいてモデルの応答をカスタマイズできます。

チューニングの詳細については、[公式ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/tuning) を参照してください。

## セットアップ

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import json
import time

import evaluate
import pandas as pd
from google import genai
from google.cloud import bigquery
from google.genai import types
from IPython.display import Markdown

In [ ]:
%load_ext google.cloud.bigquery

REGION = "global"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]
BUCKET_NAME = PROJECT_ID
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [ ]:
!gcloud storage ls --buckets | grep -w $BUCKET_URI || gcloud storage buckets create --project=$PROJECT_ID --location=$REGION $BUCKET_URI

## トレーニング データ


このノートブックでは、Gen AI SDK を使用して StackOverflow の質問と回答のデータセットで Gemini をチューニングします。
最初のステップとして、BigQuery パブリック データセットの StackOverflow データに対してクエリを実行し、`python` タグが付いた質問と、2020-01-01 以降の `accepted`（承認済み）の回答のみに絞り込みます。

データセットは 1,000 件に制限し、そのうち 800 件を LLM のチューニングに使用し、残りをチューニング済みモデルの評価に使用します。
2 番目のステップでは、チューニング ジョブが取り込めるように、データセットを 1 行に 1 件の例を含む JSONL 形式に変換します。

次に、クエリを実行してデータセットを DataFrame `df` に集約します。

In [ ]:
%%bigquery df

SELECT CONCAT(q.title, q.body) as input_text, a.body AS output_text
FROM
    `bigquery-public-data.stackoverflow.posts_questions` q
JOIN
    `bigquery-public-data.stackoverflow.posts_answers` a
ON
    q.accepted_answer_id = a.id
WHERE
    q.accepted_answer_id IS NOT NULL AND
    REGEXP_CONTAINS(q.tags, "python") AND
    a.creation_date >= "2020-01-01"
LIMIT
    1000

In [ ]:
df.head()

`parts` の下の `input_text` 列は StackOverflow ユーザーが実際に投稿した質問に対応し、`output_text` 列は正解の回答に対応しています。この 1,000 件の質問と回答のペアのデータセットから、次の形式で 1 行に 1 件の例を含む JSONL ファイルを生成する必要があります。

```python
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": input_text,
          ...
        }
      ],
    },
    {
      "role": "model",
      "parts": [
        {
          "text": output_text,
          ...
        }
      ],
    },
  ]
}
```

これは Gemini 3.5 Flash モデルをチューニングするために必要な形式の例です。

使用できるその他のフィールドについては、[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/tuning/supervised-tuning/prepare-data) を参照してください。

Gemini のチューニングには、少なくとも 100 〜 500 件の例を用意することをお勧めします。データセットで提供する例が多いほど、結果は良くなります。トレーニング データセットの例の数に上限はありません。このラボでは 800 件を使用します。
可能であれば、検証データセットも用意してください。検証データセットはチューニング ジョブの有効性を測定するのに役立ちます。検証データセットは最大 256 件の例をサポートしています。

まず、データをトレーニング用と評価用に分割しましょう。

In [ ]:
# 80/20 でデータを分割
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
evaluation = df.iloc[split_idx:]
print("train size:", len(train))
print("eval size:", len(evaluation))

In [ ]:
train.head()

チューニングを行うには、トレーニング データと評価データをまず JSONL 形式に変換する必要があります。そのために、次の 2 つのヘルパー関数を用意しています。
1 つ目は、単一の `input_text` と `output_text` のレコードを Gemini で必要な JSONL 形式に変換します。

In [ ]:
def format_for_gemini(input_text, output_text):
    return json.dumps(
        {
            "contents": [
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": input_text,
                        }
                    ],
                },
                {
                    "role": "model",
                    "parts": [
                        {
                            "text": output_text,
                        }
                    ],
                },
            ]
        }
    )

2 つ目のヘルパー関数は、データをファイルにエクスポートします。

In [ ]:
def export_tuning_data(file_name, df):
    with open(file_name, "a") as file:
        for row in df.iterrows():
            jsonline = format_for_gemini(
                row[1]["input_text"],
                row[1]["output_text"],
            )
            file.write(jsonline)
            file.write("\n")

トレーニング ファイルと評価ファイルを作成しましょう。

In [ ]:
training_data_filename = "tune_data_stack_overflow_python_qa.jsonl"
evaluation_data_filename = "evaluation_data_stack_overflow_python_qa.jsonl"

!test -f $training_data_filename    && rm $training_data_filename
!test -f $evaluation_data_filename  && rm $evaluation_data_filename

export_tuning_data(training_data_filename, train)
export_tuning_data(evaluation_data_filename, evaluation)

その後、ローカル ファイルを GCS にエクスポートして、Agent Platform のチューニング ジョブで使用できるようにします。

In [ ]:
!gcloud storage cp $training_data_filename   $BUCKET_URI
!gcloud storage cp $evaluation_data_filename $BUCKET_URI

ファイルが Google Cloud Storage バケットに正常に転送されたことを確認できます。

In [ ]:
TRAINING_DATA_URI = f"{BUCKET_URI}/{training_data_filename}"
EVALUATION_DATA_URI = f"{BUCKET_URI}/{evaluation_data_filename}"

!gcloud storage ls -a -l $TRAINING_DATA_URI
!gcloud storage ls -a -l $EVALUATION_DATA_URI

### モデルのチューニング
それでは、モデルのチューニングを開始しましょう。[Google Gen AI SDK を使用してチューニング ジョブを送信](https://googleapis.github.io/python-genai/#tune) します。
これには約 30 分かかります。

In [ ]:
client = genai.Client(vertexai=True, location="us-central1")

base_model = "gemini-3.5-flash"

training_dataset = types.TuningDataset(
    gcs_uri=TRAINING_DATA_URI,
)
evaluation_dataset = types.TuningValidationDataset(gcs_uri=EVALUATION_DATA_URI)

sft_tuning_job = client.tunings.tune(
    base_model=base_model,
    training_dataset=training_dataset,
    config=types.CreateTuningJobConfig(
        epoch_count=1,
        validation_dataset=evaluation_dataset,
        tuned_model_display_name="stackoverflow_tuned_gemini",
    ),
)

running_states = {"JOB_STATE_PENDING", "JOB_STATE_RUNNING"}

# ジョブの完了をポーリング
while sft_tuning_job.state in running_states:
    print(sft_tuning_job.state)
    sft_tuning_job = client.tunings.get(name=sft_tuning_job.name)
    time.sleep(10)

print(sft_tuning_job.tuned_model_display_name)
print(sft_tuning_job.name)
print(sft_tuning_job.experiment)

## Agent Platform Model レジストリからのチューニング済みモデルの取得


チューニング ジョブが完了すると、Agent Platform 上でモデルが利用可能になります。次のセルでは、チューニング済みモデルを一覧表示する方法を示します。

In [ ]:
for model in client.models.list(config={"page_size": 10, "query_base": False}):
    print(model)

予測を取得してみましょう。Gen AI SDK 経由でチューニング済みモデルにプロンプトを送信できます。<br>
以下のプロンプトは自由に変更してください。

In [ ]:
sft_tuning_job.tuned_model.endpoint

In [ ]:
# 推論のためにロケーションをマルチリージョンに切り替え
client = genai.Client(vertexai=True, location="us")

PROMPT = """
How can I store my TensorFlow checkpoint on Google Cloud Storage?

Python example:

"""

response = client.models.generate_content(
    model=sft_tuning_job.tuned_model.endpoint,
    contents=PROMPT,
)

Markdown(response.text)

## 手動評価

モデルのパフォーマンスを把握するには、評価を行うことが不可欠です。評価は、F1、BLEU、ROUGE などの評価指標を使用して自動化された方法で実施できます。また、人間による評価手法を活用することもできます。人間による評価手法では、人間に LLM の回答の品質を評価してもらいます。これはクラウドソーシングを利用したり、専門家に回答を評価してもらったりすることで行われます。一般的な人間による評価指標には、流暢さ（fluency）、一貫性（coherence）、関連性（relevance）、情報量（informativeness）などがあります。モデルのパフォーマンスを的確に把握するためには、多くの場合、複数の評価指標を組み合わせて選択することが望ましいです。


数ある指標の中から、ここでは 2 つのテキストが同じ意味を持っているかどうかを（自動的かつおおまかに）測定する以下の 2 つの指標を計算します。
- [BLEU](https://en.wikipedia.org/wiki/BLEU) 評価指標は**適合率（precision）**の一種であり、生成された文の中の $n$-gram が参照文の $n$-gram と一致する割合を測定します。スコアは 0 から 1 の範囲で、文が類似しているほど高くなります。BLEU1 は 1-gram（ユニグラム）のみを考慮し、BLEU2 は 2-gram（バイグラム）を考慮します。

- [ROUGE](https://en.wikipedia.org/wiki/ROUGE_(metric)) 評価指標は**再現率（recall）**の一種であり、参照文の中の $n$-gram が生成された文の $n$-gram と一致する割合を測定します。スコアは 0 から 1 の範囲で、文が類似しているほど高くなります。ROUGE1 は 1-gram（ユニグラム）のみを考慮し、ROUGE2 は 2-gram（バイグラム）を考慮します。


スコアの計算には [evaluate](https://github.com/huggingface/evaluate/tree/main) を使用します。
ノートブックの前半で、トレーニング データセットと評価データセットを作成しました。ここでは評価データの一部を使用します。質問を使用してチューニング済みモデルから応答を取得し、元の回答を参照用（正解）として使用します。
- **Candidates（候補）**: チューニング済みモデルによって生成された回答。
- **References（参照）**: 比較に使用する元の正解の回答。

In [ ]:
# 使用する行数は変更可能です
EVAL_ROWS = 60
INPUT_LIMIT = 10000  # 文字数
evaluation = evaluation[evaluation.input_text.apply(len) <= INPUT_LIMIT]
evaluation = evaluation.head(EVAL_ROWS)
evaluation.head()

下のセルの関数は、`evaluation.input_text` を使用してチューニング済みモデルにクエリを実行し、モデルの回答と並べて正解データを `evaluation.output_text` として DataFrame に保存します（これには約 5 分かかります）。

In [ ]:
def create_eval_data(model_endpoint, evaluation):
    model_answers = []

    for prompt in evaluation.input_text:
        response = client.models.generate_content(
            model=model_endpoint, contents=prompt
        )
        model_answers.append(response.text)
    eval_df = pd.DataFrame(
        {"candidate": model_answers, "reference": evaluation.output_text}
    )
    mask = eval_df.candidate == ""
    return eval_df[~mask]

In [ ]:
eval_df = create_eval_data(sft_tuning_job.tuned_model.endpoint, evaluation)

In [ ]:
eval_df.head()

次のセルの関数は、1-gram の BLEU および ROUGE スコアを計算します。すべての参照回答とチューニング済みモデルによって生成された回答全体でこれらのスコアを平均し、モデルのパフォーマンス指標となるスコアを出力します。

In [ ]:
def compute_scores(eval_data):
    predictions = eval_data.candidate.tolist()
    references = eval_data.reference.tolist()
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")
    rouge_value = rouge.compute(predictions=predictions, references=references)[
        "rouge1"
    ]
    bleu_value = bleu.compute(predictions=predictions, references=references)[
        "bleu"
    ]
    return {"rouge": rouge_value, "bleu": bleu_value}

In [ ]:
compute_scores(eval_df)

モデルの 2 つのバージョン（異なるデータ量やトレーニング ステップ数でチューニングされたものなど）がある場合、スコアを比較してどちらが優れているかを判断できます。ただし、これらの自動指標は人間による評価の非常におおまかな代替にすぎない点に留意してください。

## 自動評価


最後に、Agent Platform のチューニング ジョブはモデルのチューニングおよび評価の指標を収集してレポートすることに触れておきます。これらは、Agent Platform Studio のチューニング セクションでチューニング済みモデル名をクリックすることで、Agent Platform Experiments で可視化できます。
計算される指標の説明は以下のとおりです。


#### モデル チューニング指標

モデル チューニング ジョブは、`gemini-3.5-flash` に対する以下のチューニング指標を自動的に収集します。

* `/train_total_loss`: トレーニング ステップにおけるチューニング データセットの損失（Loss）。
* `/train_fraction_of_correct_next_step_preds`: トレーニング ステップにおけるトークン正解率。単一の予測は一連のトークンで構成されます。この指標は、予測されたトークンがチューニング データセット内の正解と比較してどれだけ正確であるかを測定します。
* `/train_num_predictions`: トレーニング ステップにおける予測トークン数。


#### モデル検証指標:

このラボで行ったように評価データセットを渡すことで、gemini-3.5-flash に対する以下の検証指標を収集するようにモデル チューニング ジョブを構成できます。

* `/eval_total_loss`: 検証ステップにおける検証データセットの損失（Loss）。
* `/eval_fraction_of_correct_next_step_preds`: 検証ステップにおけるトークン正解率。単一の予測は一連のトークンで構成されます。この指標は、予測されたトークンが検証データセット内の正解と比較してどれだけ正確であるかを測定します。
* `/eval_num_predictions`: 検証ステップにおける予測トークン数。


指標の可視化は、モデル チューニング ジョブの完了後に利用可能になります。
チューニング ジョブの作成時に検証データセットを指定しなかった場合、チューニング指標の可視化のみが利用可能となります。

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.